# 02 — Data Cleaning, Alignment, and Indicator Construction (θ)

This notebook implements:

US3 — Data Cleaning & Alignment  
US4 — Labour Market Imbalance Indicator Construction

The objective is to construct a consistent, transparent, and reproducible
monthly labour market dataset suitable for:

• Descriptive analysis  
• Diagnostic analysis  
• Predictive forecasting  
• Scenario modeling  

---

Compute labour market imbalance indicator:
   
   θₜ = Vₜ / Uₜ

   where:
   - Vₜ = job vacancies (persons)
   - Uₜ = unemployed persons

To ensure measurement consistency:

- Unemployment is converted from thousands to persons.
- Vacancies are already expressed in persons.

---

## COVID Structural Shock Treatment

Following supervisory guidance:

• The April–September 2020 period is retained for descriptive and diagnostic analysis.
• The same period is excluded from predictive modeling to preserve trend stability.

Two datasets will therefore be exported:

1. `final_dataset_full.csv`
2. `final_dataset_modeling.csv`


In [8]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)


In [9]:
# Load cleaned intermediate datasets
lfs = pd.read_csv("../1_data/processed/lfs_clean.csv")
vac = pd.read_csv("../1_data/processed/jvws_clean.csv")

print("LFS shape:", lfs.shape)
print("VAC shape:", vac.shape)

display(lfs.head())
display(vac.head())


LFS shape: (117, 2)
VAC shape: (111, 2)


,month,unemployment
0,2015-04-01,1337.2
1,2015-05-01,1324.7
2,2015-06-01,1326.6
3,2015-07-01,1336.0
4,2015-08-01,1358.3


,month,vacancies
0,2015-04-01,440425
1,2015-05-01,407040
2,2015-06-01,398910
3,2015-07-01,376570
4,2015-08-01,369190


In [10]:
# Ensure month column is datetime
lfs['month'] = pd.to_datetime(lfs['month'])
vac['month'] = pd.to_datetime(vac['month'])

print("LFS date range:", lfs['month'].min(), "to", lfs['month'].max())
print("VAC date range:", vac['month'].min(), "to", vac['month'].max())


LFS date range: 2015-04-01 00:00:00 to 2024-12-01 00:00:00
VAC date range: 2015-04-01 00:00:00 to 2024-12-01 00:00:00


In [11]:
# Merge on month
df = pd.merge(
    lfs,
    vac,
    on="month",
    how="left",
    validate="one_to_one"
)

print("Merged shape:", df.shape)
display(df.head())


Merged shape: (117, 3)


,month,unemployment,vacancies
0,2015-04-01,1337.2,440425.0
1,2015-05-01,1324.7,407040.0
2,2015-06-01,1326.6,398910.0
3,2015-07-01,1336.0,376570.0
4,2015-08-01,1358.3,369190.0


In [12]:
# Check duplicates
duplicates = df.duplicated(subset=['month']).sum()
print("Duplicate months:", duplicates)

# Missing values
print("Missing values by column:")
display(df.isna().sum())


Duplicate months: 0
Missing values by column:


month           0
unemployment    0
vacancies       6
dtype: int64

In [13]:
# Unemployment is stored in thousands → convert to persons
df_full['Unemployment_persons'] = df_full['Unemployment'] * 1000

display(df_full[['month', 'Unemployment', 'Unemployment_persons']].head())


,month,Unemployment,Unemployment_persons
0,2015-04,1337.2,1337200.0
1,2015-05,1324.7,1324700.0
2,2015-06,1326.6,1326600.0
3,2015-07,1336.0,1336000.0
4,2015-08,1358.3,1358300.0


In [21]:
# Compute labour market imbalance indicator
df_full['theta'] = np.where(
    df_full['Unemployment_persons'] > 0,
    df_full['vacancies'] / df_full['Unemployment_persons'],
    np.nan
)

print("Theta summary:")
print(df_full['theta'].describe())


Theta summary:
count    111.000000
mean       0.474081
std        0.186912
min        0.241134
25%        0.329083
50%        0.437226
75%        0.537656
max        0.976705
Name: theta, dtype: float64


In [16]:
final_columns = [
    "month",
    "Employment",
    "Unemployment_persons",
    "vacancies",
    "theta"
]

df_full_export = df_full[final_columns].copy()
df_model_export = df_model[final_columns].copy()

display(df_full_export.head())


,month,Employment,Unemployment_persons,vacancies,theta
0,2015-04,17819.4,1337200.0,440425.0,0.329364
1,2015-05,17843.7,1324700.0,407040.0,0.307270
2,2015-06,17836.7,1326600.0,398910.0,0.300701
3,2015-07,17863.3,1336000.0,376570.0,0.281864
4,2015-08,17889.0,1358300.0,369190.0,0.271803


In [17]:
# Convert month to datetime for validation ONLY
df['month'] = pd.to_datetime(df['month'])

# Sort chronologically
df = df.sort_values('month').reset_index(drop=True)

# Check start and end dates
print("Start month:", df['month'].min())
print("End month:", df['month'].max())

# Check monthly frequency
expected_months = pd.date_range(
    start=df['month'].min(),
    end=df['month'].max(),
    freq='MS'
)

actual_months = df['month']

missing_months = expected_months.difference(actual_months)

print("\nMissing months:")
print(missing_months)

# Confirm one row per month
print("\nUnique months:", df['month'].nunique())
print("Total rows:", df.shape[0])


Start month: 2015-04-01 00:00:00
End month: 2024-12-01 00:00:00

Missing months:
DatetimeIndex([], dtype='datetime64[ns]', freq='MS')

Unique months: 117
Total rows: 117


In [18]:
# Convert month to pandas Period (monthly)
df['month'] = pd.to_datetime(df['month']).dt.to_period('M')

# Check result
df.head()


,month,unemployment,vacancies
0,2015-04,1337.2,440425.0
1,2015-05,1324.7,407040.0
2,2015-06,1326.6,398910.0
3,2015-07,1336.0,376570.0
4,2015-08,1358.3,369190.0


### Handling of Missing Job Vacancy Data

The Job Vacancy and Wage Survey (JVWS) contains missing observations for the period
April 2020 to September 2020. This gap corresponds to the initial COVID-19 shock,
during which vacancy data collection and publication were disrupted.

To preserve the integrity of the time series and avoid introducing undocumented
assumptions, these observations were retained as missing values (NA). No imputation
or smoothing was applied at this stage. The handling of these missing values is
explicitly documented and will be addressed in later analytical phases if required.



Outlier inspection was conducted on the θ indicator using interquartile range (IQR)
bounds. Extreme values were observed primarily during periods of sharp changes in
unemployment and job vacancies, particularly around the COVID-19 shock and post-
pandemic recovery.

These values were retained, as they reflect meaningful economic conditions rather
than data errors. No outlier removal or smoothing was applied at this stage.


## Indicator Validation — Outlier and Integrity Checks

This section validates the constructed imbalance indicator (θ).

The goal is not to mechanically remove extreme values,
but to determine whether they represent:

• Data errors  
• Structural shocks  
• Legitimate economic regimes  

Outliers are detected using the IQR method.


In [32]:
# IQR method
Q1 = df_full['theta'].quantile(0.25)
Q3 = df_full['theta'].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

print("Lower bound:", lower_bound)
print("Upper bound:", upper_bound)

outliers = df[
    (df_full['theta'] < lower_bound) |
    (df_full['theta'] > upper_bound)
]

print("Number of outliers:", len(outliers))
display(outliers[['month', 'unemployment', 'vacancies']])



Lower bound: 0.016224847368886686
Upper bound: 0.850514510666321
Number of outliers: 7


,month,unemployment,vacancies
83,2022-03,1114.9,984390.0
84,2022-04,1116.5,979550.0
85,2022-05,1084.2,987705.0
86,2022-06,1027.4,988430.0
87,2022-07,1007.1,983640.0
88,2022-08,1083.9,945520.0
89,2022-09,1066.6,946285.0


## Final Clean Dataset Export

This dataset represents the fully cleaned, aligned, and validated
labour market dataset.

Features:

• Monthly alignment complete  
• Units standardized  
• θ constructed correctly  
• Outliers inspected and documented  
• No artificial interpolation applied  

This dataset will be used for:
- Descriptive analysis
- Diagnostic analysis
- Predictive modeling (with filtering done in Phase 4)


In [ ]:
# Select final required columns
final_columns = [
    "month",
    "Employment",
    "Unemployment_persons",
    "vacancies",
    "theta"
]

# Use df_full (contains Employment, Unemployment_persons, theta) rather than df
df_final = df_full[final_columns].copy()

# Sort chronologically
df_final = df_final.sort_values("month").reset_index(drop=True)

print("Final dataset shape:", df_final.shape)
display(df_final.head())

# Export
df_final.to_csv("../1_data/processed/final_dataset_clean.csv", index=False)



Final dataset shape: (117, 5)


,month,Employment,Unemployment_persons,vacancies,theta
0,2015-04,17819.4,1337200.0,440425.0,0.329364
1,2015-05,17843.7,1324700.0,407040.0,0.307270
2,2015-06,17836.7,1326600.0,398910.0,0.300701
3,2015-07,17863.3,1336000.0,376570.0,0.281864
4,2015-08,17889.0,1358300.0,369190.0,0.271803


✅ final_dataset_clean.csv exported successfully.
